# 04 — Model comparison + ablation analysis

Prereqs: `02_train_dan.ipynb` and `05_train_convnext.ipynb` produced `runs/dan_fer2013/best.pth` and `runs/convnext_fer2013/best.pth`. Optionally, the three ablation configs ran via:
```
for cfg in dan_no_sampler dan_no_augment dan_no_imagenet; do
  python -m src.train --config configs/ablations/${cfg}.yaml
  python -m src.eval --config configs/ablations/${cfg}.yaml --ckpt runs/${cfg/dan_/dan_fer2013_}/best.pth
done
```

Cells:
- **A** — Comparison table (DAN, ConvNeXt-Tiny, +TTA, ensemble, ablations)
- **B** — Side-by-side normalized confusion matrices
- **C** — Per-class recall delta heatmap (where does each model win?)
- **D** — Failure-mode gallery (top-confident-wrong per model)
- **E** — Agreement matrix (orthogonality story for ensembling)
- **F** — Cross-dataset eval (skipped if RAF-DB unavailable)

In [ ]:
%cd /content/fer
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader

from src.data import CLASSES, NUM_CLASSES, FER2013Dataset, RAFDBDataset, build_transforms
from src.eval import build_tta_transform, collect_predictions, confusion_matrix, per_class_metrics, _logits
from src.models import build_model
from src.ensemble import ensemble_eval

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## Cell A — Comparison table
Reads `runs/<exp>/eval/metrics.json` and `eval_tta/metrics.json` from every available run.

In [ ]:
EXPECTED_RUNS = [
    ('DAN',                       'runs/dan_fer2013'),
    ('ConvNeXt-Tiny',             'runs/convnext_fer2013'),
    ('DAN  (no sampler)',         'runs/dan_fer2013_no_sampler'),
    ('DAN  (no augment)',         'runs/dan_fer2013_no_augment'),
    ('DAN  (no ImageNet init)',   'runs/dan_fer2013_no_imagenet'),
]

rows = []
for label, run_dir in EXPECTED_RUNS:
    for tag, sub in [('', 'eval'), ('+TTA', 'eval_tta')]:
        p = Path(run_dir) / sub / 'metrics.json'
        if not p.exists(): continue
        m = json.loads(p.read_text())
        rows.append({'config': f'{label}{tag}', 'WAR': m['war'], 'UAR': m['uar'], 'n': m['n_samples']})

# Optional: pull ensemble result if it exists.
for ens_dir in ['runs/ensemble_dan_convnext', 'runs/ensemble_dan_convnext_tta']:
    p = Path(ens_dir) / 'metrics.json'
    if p.exists():
        m = json.loads(p.read_text())
        tag = ' (TTA)' if m.get('tta') else ''
        rows.append({'config': f'Ensemble DAN+ConvNeXt{tag}', 'WAR': m['war'], 'UAR': m['uar'], 'n': m['n_samples']})

df = pd.DataFrame(rows).sort_values('WAR', ascending=False).reset_index(drop=True)
df

## Cell B — Side-by-side normalized confusion matrices

In [ ]:
MODELS_FOR_CM = [
    ('DAN',           'dan',           'runs/dan_fer2013/best.pth'),
    ('ConvNeXt-Tiny', 'convnext_tiny', 'runs/convnext_fer2013/best.pth'),
]
TRANSFORM = build_transforms(train=False, image_size=224)
test_ds = FER2013Dataset('data/fer2013', split='test', transform=TRANSFORM)
test_loader = DataLoader(test_ds, batch_size=128, num_workers=2)

def load_model(model_name, ckpt_path):
    m = build_model(model_name).to(device)
    state = torch.load(ckpt_path, map_location=device, weights_only=False)
    sd = state.get('model', state.get('model_state_dict', state.get('state_dict', state)))
    sd = {k.removeprefix('module.'): v for k, v in sd.items()}
    m.load_state_dict(sd, strict=False)
    return m.eval()

predictions = {}  # name -> (pred, true)
available = [(n, mn, p) for n, mn, p in MODELS_FOR_CM if Path(p).exists()]

fig, axes = plt.subplots(1, max(1, len(available)), figsize=(7 * max(1, len(available)), 6), squeeze=False)
for ax, (name, model_name, ckpt) in zip(axes[0], available):
    model = load_model(model_name, ckpt)
    pred, true = collect_predictions(model, test_loader, device)
    predictions[name] = (pred, true)
    cm = confusion_matrix(pred, true, NUM_CLASSES)
    cm_norm = cm.astype(np.float64) / cm.sum(axis=1, keepdims=True).clip(min=1)
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASSES, rotation=45, ha='right')
    ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASSES)
    war = (pred == true).mean()
    ax.set_title(f'{name}  (WAR={war:.3f})')
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, f'{cm_norm[i,j]:.2f}', ha='center', va='center', fontsize=7,
                    color='white' if cm_norm[i,j] > 0.5 else 'black')
    del model
    if device.type == 'cuda': torch.cuda.empty_cache()

fig.colorbar(im, ax=axes[0], shrink=0.8)
plt.tight_layout(); plt.show()

## Cell C — Per-class recall delta
Bars show `recall(ConvNeXt) - recall(DAN)` per class — positive = ConvNeXt wins on that emotion.

In [ ]:
if {'DAN', 'ConvNeXt-Tiny'}.issubset(predictions):
    pred_dan, true = predictions['DAN']
    pred_cnx, _    = predictions['ConvNeXt-Tiny']
    delta = []
    for c in range(NUM_CLASSES):
        m = true == c
        if m.sum() == 0: delta.append(0.0); continue
        delta.append(float((pred_cnx[m] == c).mean() - (pred_dan[m] == c).mean()))
    fig, ax = plt.subplots(figsize=(8, 4))
    colors = ['#2ca02c' if d >= 0 else '#d62728' for d in delta]
    ax.bar(CLASSES, delta, color=colors)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_ylabel('recall(ConvNeXt) - recall(DAN)')
    ax.set_title('Per-class recall delta on FER-2013 PrivateTest')
    for i, d in enumerate(delta):
        ax.text(i, d + (0.005 if d >= 0 else -0.015), f'{d:+.3f}', ha='center', fontsize=9)
    plt.tight_layout(); plt.show()
else:
    print('Need both DAN and ConvNeXt predictions; run Cell B first.')

## Cell D — Failure-mode gallery
Top-10 confident-wrong samples per model. If both models fail on the same images → ensemble unlikely to help.

In [ ]:
def confident_wrong(model_name, ckpt_path, k=10):
    model = load_model(model_name, ckpt_path)
    wrong = []
    idx = 0
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(device)
            out = model(imgs)
            logits = out[0] if isinstance(out, (tuple, list)) else out
            probs = F.softmax(logits, dim=1).cpu()
            conf, pred = probs.max(dim=1)
            for i in range(len(labels)):
                if pred[i].item() != labels[i].item():
                    wrong.append((conf[i].item(), int(labels[i]), int(pred[i]), idx + i))
            idx += len(labels)
    del model
    if device.type == 'cuda': torch.cuda.empty_cache()
    wrong.sort(reverse=True)
    return wrong[:k]

for name, model_name, ckpt in available:
    print(f'\n=== {name} confident-wrong ===')
    cw = confident_wrong(model_name, ckpt, k=10)
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    for ax, (cf, t, p, i) in zip(axes.flat, cw):
        rel, _ = test_ds.samples[i]
        ax.imshow(Image.open(test_ds.root / rel)); ax.axis('off')
        ax.set_title(f'{CLASSES[t]} -> {CLASSES[p]} ({cf:.2f})', fontsize=9)
    plt.suptitle(name); plt.tight_layout(); plt.show()

## Cell E — Agreement matrix
Shows how often DAN's prediction matches ConvNeXt's. High disagreement → ensemble can recover errors.

In [ ]:
if {'DAN', 'ConvNeXt-Tiny'}.issubset(predictions):
    pred_dan, true = predictions['DAN']
    pred_cnx, _    = predictions['ConvNeXt-Tiny']
    agree = (pred_dan == pred_cnx).mean()
    both_correct = ((pred_dan == true) & (pred_cnx == true)).mean()
    only_dan_correct = ((pred_dan == true) & (pred_cnx != true)).mean()
    only_cnx_correct = ((pred_dan != true) & (pred_cnx == true)).mean()
    both_wrong = ((pred_dan != true) & (pred_cnx != true)).mean()
    print(f'Overall agreement DAN ↔ ConvNeXt: {agree:.3f}')
    print(f'  both correct:        {both_correct:.3f}')
    print(f'  only DAN correct:    {only_dan_correct:.3f}  <- ensemble could recover')
    print(f'  only ConvNeXt right: {only_cnx_correct:.3f}  <- ensemble could recover')
    print(f'  both wrong:          {both_wrong:.3f}  <- ensemble cannot help here')
    print(f'\nEnsemble upper bound (oracle picks best): {1 - both_wrong:.3f}')

## Cell F — Cross-dataset evaluation (if RAF-DB available)
Train on FER-2013, eval on RAF-DB test (and vice versa). Both label spaces are aligned by `scripts/prepare_rafdb.py`.

In [ ]:
if not Path('data/rafdb/manifest.csv').exists():
    print('RAF-DB not available; skipping cross-dataset eval.')
else:
    rows = []
    for name, model_name, ckpt in available:
        m = load_model(model_name, ckpt)
        for ds_name, ds_cls, root in [('rafdb', RAFDBDataset, 'data/rafdb')]:
            ds = ds_cls(root, split='test', transform=TRANSFORM)
            loader = DataLoader(ds, batch_size=128, num_workers=2)
            pred, true = collect_predictions(m, loader, device)
            war = float((pred == true).mean())
            cm = confusion_matrix(pred, true, NUM_CLASSES)
            uar = float(np.mean([m['recall'] for m in per_class_metrics(cm)]))
            rows.append({'model': name, 'eval_set': ds_name, 'WAR': war, 'UAR': uar})
        del m
        if device.type == 'cuda': torch.cuda.empty_cache()
    pd.DataFrame(rows)